# Compare a `post_isr_image` with a `preliminary_visit_image` (DRP stage1, step1a)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-29 : stage 1 (display of the postISRCCD image, RA increasing to the left)
- last update : 2026-11-06 : renamed (previous name `nb2026-11-05_PlotPostISRCCD_DRPstage1-single-visit.ipynb`), now compares the two images of the same detector, one visit per band

## What is compared

Both images are produced by **stage 1, step 1a** (`DRP.yaml#stage1-single-visit`, per detector) :

- `post_isr_image` : the CCD image after instrument signature removal (ISR)
- `preliminary_visit_image` : the image of the same detector after the calibration of the step (astrometry, photometry, background), see the step1a diagram below

For each band (u, g, r, i, z, y) present in the collection, one visit and one detector are picked, and the two images are displayed side by side, in the same pixel frame, with the `single_visit_star` sources overlaid on both.

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## Setup

In [ ]:
%matplotlib inline

import os
import logging
import warnings
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval
from IPython.display import display

import lsst.geom as geom
from lsst.daf.butler import Butler

import showplots_utils
importlib.reload(showplots_utils)   # pick up any edit of the helper module without restarting the kernel
from showplots_utils import show_diagram

In [ ]:
# Turn the Butler.collections FutureWarning into an error to catch outdated API usage early
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

logging.basicConfig(
    level=logging.WARNING,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)

butler = Butler('main')

In [ ]:
username = os.getenv('USER')
RUN_NAME = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'
collection_path = f'u/{username}/LSSTCam/{RUN_NAME}/stage1'

BANDS = 'ugrizy'          # one image pair per band, when the band is present in the collection
DETECTOR_ID = None        # None = first detector available for each band
SHOW_SOURCES = True       # overlay the single_visit_star sources on both images
STAR_RADIUS_DEG = 0.3     # only the catalog stars closer than this to the detector center are projected

DIAGRAM_DIR = Path('../../../../notebooks/fig_diagrams')
FIGDIR = Path('figs_ComparePostISRvsPreliminary')
FIGDIR.mkdir(parents=True, exist_ok=True)

print('collection :', collection_path)

In [ ]:
show_diagram(DIAGRAM_DIR / 'pipeline_drp_pipe_LSSTCam_DRP_step1a-single-visit-detectors.png', title='DRP step1a/detectors')

## Find one image pair per band

A (visit, detector) pair is kept when it has a `post_isr_image`, a `preliminary_visit_image` and a `single_visit_star` catalog. The `post_isr_image` is labelled by `exposure` and the others by `visit` ; for LSSTCam the two identifiers are equal.

In [ ]:
def query_refs(dataset_type):
    '''All the dataset references of a dataset type in the collection.'''
    return list(
        butler.registry.queryDatasets(
            datasetType=dataset_type,
            collections=collection_path,
            findFirst=False,
        )
    )


def dval(ref, key):
    '''Value of a dimension in the dataId of a reference (None if absent).'''
    try:
        return ref.dataId[key]
    except KeyError:
        return None


refs_isr = query_refs('post_isr_image')
refs_pvi = query_refs('preliminary_visit_image')
refs_star = query_refs('single_visit_star')

for name, refs in (('post_isr_image', refs_isr),
                   ('preliminary_visit_image', refs_pvi),
                   ('single_visit_star', refs_star)):
    print(f'{name:25s} : {len(refs)} dataset(s), dimensions {list(refs[0].dataId.dimensions.names) if refs else None}')
    if not refs:
        raise RuntimeError(f'No {name} in {collection_path}')

isr_by_key = {(dval(r, 'exposure'), dval(r, 'detector')): r for r in refs_isr}
pvi_by_key = {(dval(r, 'visit'), dval(r, 'detector')): r for r in refs_pvi}
star_by_key = {(dval(r, 'visit'), dval(r, 'detector')): r for r in refs_star}   # detector is None for a visit-level catalog
star_visits = {k[0] for k in star_by_key}

common = sorted(k for k in isr_by_key.keys() & pvi_by_key.keys() if k[0] in star_visits)
print(f'(visit, detector) pairs with the 3 datasets : {len(common)}')

In [ ]:
# Band of each visit : visit -> physical_filter -> band
filter_to_band = {
    rec.name: rec.band
    for rec in butler.registry.queryDimensionRecords('physical_filter', instrument='LSSTCam')
}
visit_to_band = {
    rec.id: filter_to_band.get(rec.physical_filter)
    for rec in butler.registry.queryDimensionRecords(
        'visit', instrument='LSSTCam',
        datasets='preliminary_visit_image', collections=collection_path,
    )
}

# One (visit, detector) pair per band
selection = {}
for band in BANDS:
    keys = [k for k in common
            if visit_to_band.get(k[0]) == band and (DETECTOR_ID is None or k[1] == DETECTOR_ID)]
    if keys:
        selection[band] = keys[0]

display(pd.DataFrame(
    [dict(band=b, visit=k[0], detector=k[1]) for b, k in selection.items()]
).set_index('band'))

missing = [b for b in BANDS if b not in selection]
if missing:
    print('Bands without a complete image pair :', missing)

## Display functions

- each image has its own asinh / zscale stretch (the two images do not have the same pixel values)
- the sources are projected to pixels with the WCS of the `preliminary_visit_image`, and the same pixel positions are drawn on both images (both images share the detector pixel frame)
- the axes are flipped, using the same WCS, so that RA increases to the left (East left) and Dec upward (North up) ; a yellow E/N compass allows to check the orientation

In [ ]:
def to_pandas(table):
    '''Catalog (astropy / arrow table) to DataFrame.'''
    if isinstance(table, pd.DataFrame):
        return table
    return table.to_pandas()


def find_radec_columns(df):
    '''Names of the (RA, Dec) columns of a catalog, or None.'''
    names = {c.lower(): c for c in df.columns}
    for cra, cdec in [('ra', 'dec'), ('coord_ra', 'coord_dec'), ('raj2000', 'decj2000')]:
        if cra in names and cdec in names:
            return names[cra], names[cdec]
    return None


def star_pixel_positions(df, wcs, bbox):
    '''Pixel positions (x, y) of the catalog stars falling on the detector (None if no RA, Dec columns).'''
    cols = find_radec_columns(df)
    if cols is None:
        print('No RA, Dec columns found in the catalog :', df.columns.tolist())
        return None
    ra = df[cols[0]].to_numpy(dtype=float)
    dec = df[cols[1]].to_numpy(dtype=float)
    if np.nanmax(np.abs(ra)) <= 2 * np.pi + 0.1:   # radians -> degrees
        ra, dec = np.degrees(ra), np.degrees(dec)

    # Keep only the stars around the detector center, then project them with the WCS
    center = wcs.pixelToSky(bbox.getCenter())
    ra0, dec0 = center.getRa().asDegrees(), center.getDec().asDegrees()
    dra = (ra - ra0 + 180.0) % 360.0 - 180.0
    near = np.hypot(dra * np.cos(np.radians(dec0)), dec - dec0) < STAR_RADIUS_DEG

    xs, ys = [], []
    for r, d in zip(ra[near], dec[near]):
        p = wcs.skyToPixel(geom.SpherePoint(r, d, geom.degrees))
        xs.append(p.getX())
        ys.append(p.getY())
    xs, ys = np.array(xs), np.array(ys)

    inside = ((xs >= bbox.getMinX()) & (xs <= bbox.getMaxX())
              & (ys >= bbox.getMinY()) & (ys <= bbox.getMaxY()))
    return xs[inside], ys[inside]


def sky_orientation(wcs, bbox):
    '''Unit vectors, in pixel (x, y) space, pointing towards East and towards North.'''
    center = bbox.getCenter()
    xc, yc = center.getX(), center.getY()
    step = 100.0

    def sky_deg(x, y):
        sp = wcs.pixelToSky(geom.Point2D(x, y))
        return sp.getRa().asDegrees(), sp.getDec().asDegrees()

    ra0, dec0 = sky_deg(xc, yc)
    ra_x, dec_x = sky_deg(xc + step, yc)
    ra_y, dec_y = sky_deg(xc, yc + step)

    def d_east(ra):
        return ((ra - ra0 + 180.0) % 360.0 - 180.0) * np.cos(np.radians(dec0))

    # Jacobian : rows = (East, North) offsets in deg, columns = (x, y) pixel steps
    jac = np.array([[d_east(ra_x), d_east(ra_y)],
                    [dec_x - dec0, dec_y - dec0]]) / step
    inv = np.linalg.inv(jac)
    east = inv[:, 0] / np.hypot(*inv[:, 0])
    north = inv[:, 1] / np.hypot(*inv[:, 1])
    return east, north


def orient_east_left(ax, east, north, bbox, fontsize=16):
    '''Flip the axes so that East is on the left and North up, and draw a yellow E/N compass.'''
    if abs(east[0]) >= abs(east[1]):
        if east[0] > 0:
            ax.invert_xaxis()   # East points right -> make RA increase to the left
        if north[1] < 0:
            ax.invert_yaxis()   # North points down -> make Dec increase upward
    else:
        print('WARNING : detector rotated by ~90 deg on the sky, axis flips cannot give East-left / North-up')

    base = np.array([bbox.getMinX() + 0.15 * bbox.getWidth(), bbox.getMinY() + 0.15 * bbox.getHeight()])
    length = 0.1 * min(bbox.getWidth(), bbox.getHeight())
    for label, vec in (('E', east), ('N', north)):
        ax.annotate('', xy=base + length * vec, xytext=base,
                    arrowprops=dict(arrowstyle='->', color='yellow', lw=2))
        ax.text(*(base + 1.25 * length * vec), label, color='yellow',
                ha='center', va='center', fontsize=fontsize, fontweight='bold')

In [ ]:
def show_pair(band, key):
    '''post_isr_image and preliminary_visit_image of one (visit, detector), side by side.'''
    visit, detector = key
    exp_isr = butler.get(isr_by_key[key])
    exp_pvi = butler.get(pvi_by_key[key])

    wcs = exp_pvi.getWcs()
    bbox = exp_pvi.getBBox()
    if exp_isr.getBBox() != bbox:
        print(f'WARNING : the two images have different bounding boxes ({exp_isr.getBBox()} / {bbox}), the sources may be misplaced on the post_isr_image')

    # Sources of the visit, projected on the pixel frame of the detector
    stars = None
    if SHOW_SOURCES:
        ref_star = star_by_key.get((visit, detector)) or star_by_key.get((visit, None))
        stars = star_pixel_positions(to_pandas(butler.get(ref_star)), wcs, bbox)

    east, north = sky_orientation(wcs, bbox)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    for ax, exp, name in ((axes[0], exp_isr, 'post_isr_image'),
                          (axes[1], exp_pvi, 'preliminary_visit_image')):
        img = exp.image.array
        bb = exp.getBBox()
        extent = (bb.getMinX() - 0.5, bb.getMaxX() + 0.5, bb.getMinY() - 0.5, bb.getMaxY() + 0.5)
        norm = ImageNormalize(img, interval=ZScaleInterval(), stretch=AsinhStretch())
        ax.imshow(img, origin='lower', cmap='gray', norm=norm, extent=extent)
        if stars is not None:
            ax.scatter(stars[0], stars[1], s=70, facecolors='none', edgecolors='r', linewidths=0.8)
        ax.set_title(name, fontsize=16)
        ax.set_xlabel('x (pixel)', fontsize=14)
        ax.set_ylabel('y (pixel)', fontsize=14)
        orient_east_left(ax, east, north, bbox)

    n_stars = f', {len(stars[0])} sources' if stars is not None else ''
    fig.suptitle(f'band {band}, visit {visit}, detector {detector}{n_stars}', fontsize=18)
    fig.tight_layout()

    for ext in ('png', 'pdf'):
        fig.savefig(FIGDIR / f'compare_{band}_visit{visit}_det{detector}.{ext}', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## One image pair per band

In [ ]:
for band, key in selection.items():
    try:
        show_pair(band, key)
    except Exception as inst:
        print(type(inst)," ::: ",inst.args)     
 